# Budget experiment: what the L4 buys

Issue for Phase 7. Four epochs of pretraining at the default attention budget and four at four times that, same
seed, same data, measuring wall time and validation loss.

`ATTENTION_BUDGET` caps a batch at about 64 texts of 160 tokens whatever the card holds, which is why the first
run sat at 3.8 GB of the L4's 22.5. Nothing here trains the final model; the checkpoints are throwaway.

In [ ]:
from pathlib import Path
import os
import subprocess
REPO = Path("/content/DeepSniffing")
URL = "https://github.com/suprch4rg3d/DeepSniffing.git"
if not REPO.is_dir():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "fetch", "--depth", "1", "origin", "main"], cwd=REPO, check=True)
    subprocess.run(["git", "reset", "--hard", "FETCH_HEAD"], cwd=REPO, check=True)
os.chdir(REPO)
print(subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

In [ ]:
!pip install -q uv
import os
VENV = "/content/dsn-venv"
os.environ["VENV"] = VENV
!uv venv --python 3.13 "$VENV"
!uv pip install --python "$VENV/bin/python" poetry
os.environ["VIRTUAL_ENV"] = VENV
os.environ["PATH"] = f"{VENV}/bin:" + os.environ["PATH"]
!cd /content/DeepSniffing && poetry install --only main

In [ ]:
!python -c "import torch; assert torch.cuda.is_available(), 'no GPU: Runtime > Change runtime type > L4 GPU'; print(torch.cuda.get_device_name(0))"


In [ ]:
import shutil
free = shutil.disk_usage("/content").free / 1e9
print(f"{free:.0f} GB free")
if free < 20:
    raise RuntimeError(f"{free:.0f} GB free; ds-build needs about 20 GB")


In [ ]:
!cd /content/DeepSniffing && ds-prepare
!cd /content/DeepSniffing && ds-build

## Two runs

Both start from the same seed and see the same data. The only difference is the attention budget, which sets how
many texts fit in a batch, and the batch cap that follows from it.

In [ ]:
import json
import shutil
import subprocess
import threading
import time
from pathlib import Path
root = Path("/content/DeepSniffing")
stage = root / "output/odor/stage1"
analysis = root / "output/analysis"
analysis.mkdir(parents=True, exist_ok=True)
runs = {"64": (64 * 160 * 160, 64), "256": (256 * 160 * 160, 256)}
results = {}
for name, (budget, batch) in runs.items():
    for leftover in ("state.pt", "state.prev"):
        (stage / leftover).unlink(missing_ok=True)
    command = [
        "ds-train", "--stage", "1", "--max-epochs", "4",
        "--attention-budget", str(budget), "--batch-size", str(batch),
    ]
    print("=" * 20, name, " ".join(command))
    samples = []
    stop = threading.Event()
    def watch():
        while not stop.is_set():
            line = subprocess.run(
                ["nvidia-smi", "--query-gpu=memory.used,utilization.gpu", "--format=csv,noheader,nounits"],
                capture_output=True,
                text=True,
            ).stdout.strip()
            if line:
                samples.append(line)
            stop.wait(5)
    watcher = threading.Thread(target=watch, daemon=True)
    watcher.start()
    started = time.perf_counter()
    done = subprocess.run(command, cwd=root, capture_output=True, text=True)
    took = time.perf_counter() - started
    stop.set()
    watcher.join()
    print(done.stdout[-1200:], done.stderr[-800:])
    if done.returncode:
        raise RuntimeError(f"budget {name} exited {done.returncode}")
    history = stage / "history.csv"
    shutil.copyfile(history, analysis / f"budget_{name}_history.csv")
    rows = [line.split(",") for line in history.read_text().strip().splitlines()[1:]]
    memory = [int(s.split(",")[0]) for s in samples]
    results[name] = {
        "attention_budget": budget,
        "batch_size": batch,
        "peak_memory_mb": max(memory) if memory else None,
        "samples": samples,
        "seconds": round(took, 1),
        "seconds_per_epoch": round(took / len(rows), 1),
        "train": [float(row[1]) for row in rows],
        "validation": [float(row[2]) for row in rows],
    }
    print(name, results[name]["seconds"], "s,", results[name]["seconds_per_epoch"], "s per epoch,", results[name]["peak_memory_mb"], "MB peak")
target = analysis / "budget.json"
target.write_text(json.dumps(results, indent=1))
print("wrote", target)


In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
results = json.loads(Path("/content/DeepSniffing/output/analysis/budget.json").read_text())
fig, ax = plt.subplots(figsize=(5, 3))
for name, found in results.items():
    epochs = range(1, len(found["validation"]) + 1)
    ax.plot(epochs, found["validation"], marker="o", label=f"budget {name}, batch {found['batch_size']}")
ax.set_xlabel("epoch")
ax.set_ylabel("validation loss")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
import json
import subprocess
import zipfile
from datetime import UTC, datetime
from pathlib import Path
root = Path("/content/DeepSniffing")
results = json.loads((root / "output/analysis/budget.json").read_text())
for name, found in results.items():
    print(name, "budget", found["attention_budget"], "batch", found["batch_size"], found["seconds"], "s", found["validation"][-1], "val")
commit = subprocess.run(["git", "rev-parse", "HEAD"], cwd=root, capture_output=True, text=True).stdout.strip()
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
manifest = f"commit: {commit}\ndate: {datetime.now(UTC).isoformat(timespec='seconds')}\ngpu: {gpu}\n"
zip_name = "/content/budget-experiment.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.writestr("RUN.txt", manifest)
    zf.write(root / "output/analysis/budget.json", "budget.json")
    for name in results:
        zf.write(root / f"output/analysis/budget_{name}_history.csv", f"history_budget{name}.csv")
    zf.write(root / "notebooks/batch-budget-experiment-on-colab.ipynb", "batch-budget-experiment-on-colab.ipynb")
print("wrote", zip_name, Path(zip_name).stat().st_size, "bytes")
try:
    from google.colab import files as colab_files
    colab_files.download(zip_name)
except ImportError:
    pass